# PARADE: score and design cell-type-specific UTRs

[PARADE](https://github.com/autosome-ru/parade) (Prediction And RAtional DEsign of mRNA UTRs; [Khoroshkin et al., 2024](https://doi.org/10.1101/2024.12.31.630783)) is a LegNet convolutional model for the untranslated regions of mRNA. This toolkit wraps it as three tools:

| Tool | What it does |
|---|---|
| `parade-activity` | Predict cell-type-specific 5'/3' UTR activity across the PARADE cell-code panel |
| `parade-stability` | Predict 3' UTR mRNA stability (RNA/gDNA log-ratio) |
| `parade-gradient` | Differentiable UTR-activity objective + gradient, for gradient-based design |

This notebook walks through all three: **(A)** scoring UTRs, **(B)** the differentiable objective, and **(C)** using it to *design* UTRs whose activity is specific to one cell line over another, with a visible optimization climb for both 5'UTR and 3'UTR.

**Cell codes.** PARADE uses anonymized cell-line codes. The 5'UTR panel is `c1, c2, c4, c6, c17`; the 3'UTR panel adds `c13`. (`c2` is the only code with a public identity: HepG2.)

> **Weights.** The published checkpoints download automatically on first use from a pinned `autosome-ru/parade` commit and are cached under `PROTO_HOME`. A GPU speeds things up but everything below also runs on CPU (set `DEVICE = "cpu"`).

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Every PARADE tool is a `run_*` function plus its typed Input/Config classes.
from proto_tools.tools import (
    # --- scoring ---
    run_parade_activity, ParadeActivityInput, ParadeActivityConfig,
    run_parade_stability, ParadeStabilityInput, ParadeStabilityConfig,
    # --- differentiable design ---
    run_parade_gradient, ParadeGradientInput, ParadeGradientConfig, ParadeGradientLossTerm,
)

# Device for inference. The tool runs the model inside its own isolated environment; this only
# controls which device that worker uses. Use "cpu" if you don't have a CUDA GPU.
DEVICE = "cuda"

# parade-gradient works on relaxed nucleotide logits with columns in A,C,G,T order.
VOCAB = "ACGT"

## Tool reference

The registry can render each tool's overview and its Input/Config/Output schema, so you never have to guess field names or defaults.

In [ ]:
from proto_tools.utils.notebook_docs import (
    display_api_reference, display_available_tools, display_docs_section, display_doc_link, display_overview,
)

display_doc_link("parade")            # link to the hosted docs page
display_overview("parade")            # one-paragraph summary
display_available_tools("parade")     # the three registered tools
display_docs_section("parade", "Background")  # scientific background from the toolkit README

## Part A — Score UTRs for cell-type activity

`parade-activity` predicts an activity value **per requested cell code** for each UTR. A few things to know about the input:

- **Pick the construct type.** `construct_type="utr5"` or `"utr3"` selects the matching checkpoint and cell-code panel.
- **Cell codes are panel-specific.** Leave `cell_types` empty to get the whole panel; `c13` is 3'UTR-only.
- **Match the training length.** Upstream trained the 5'UTR model on ~50-nt and the 3'UTR model on ~240-nt (≈200–300 nt) inserts; the model accepts any length but predictions are only meaningful near those regimes. Mixed lengths in one call are batched per length group; RNA input (`U`) is accepted and mapped to DNA (`T`).

In [ ]:
# The full input/config/output schema for parade-activity:
display_api_reference("parade-activity", "input", "run_parade_activity")
display_api_reference("parade-activity", "config", "run_parade_activity")
display_api_reference("parade-activity", "output", "run_parade_activity")

In [ ]:
# Score three 50-nt 5'UTRs across the FULL 5'UTR panel (cell_types left empty -> all five codes).
utrs = [
    "GCACCATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATC",  # a GC-rich insert
    "AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA",  # a low-complexity control
    "GCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGC",
]
activity = run_parade_activity(
    ParadeActivityInput(sequences=utrs),
    ParadeActivityConfig(construct_type="utr5", device=DEVICE),
)

# `activity.results` is one item per input sequence; `result.scores` is a metrics container keyed by
# cell code. We print a plain table (no pandas dependency, so this renders even if the kernel's
# pandas/numpy are mismatched).
cols = activity.cell_types
print("sequence           | " + " | ".join(f"{c:>7}" for c in cols))
for r in activity.results:
    row = dict(r.scores.items())
    print(f"{r.sequence[:16]:<18} | " + " | ".join(f"{row[c]:7.3f}" for c in cols))

In [ ]:
# Advanced: score 3'UTRs for only a SUBSET of the panel (here the 3'UTR-only code c13 vs c2),
# with a larger GPU batch size. Note construct_type="utr3".
activity_utr3 = run_parade_activity(
    ParadeActivityInput(sequences=["GC" * 120, "AT" * 120]),  # ~240 nt: the 3'UTR training length
    ParadeActivityConfig(construct_type="utr3", cell_types=["c2", "c13"], batch_size=8, device=DEVICE),
)
for r in activity_utr3.results:
    print(r.sequence[:16], "...", {k: round(val, 3) for k, val in r.scores.items()})

### 3' UTR mRNA stability

`parade-stability` predicts an RNA/gDNA **log-ratio** for 3' UTRs — higher means more stable. There is no cell conditioning, so it returns a single value per sequence.

In [ ]:
display_api_reference("parade-stability", "input", "run_parade_stability")
display_api_reference("parade-stability", "output", "run_parade_stability")

# The stability model was trained on 186-nt 3'UTR sequences; score near that length.
stability_utrs = ["ACGT" * 46 + "AC", "TGCA" * 46 + "GT"]  # 186 nt
stability = run_parade_stability(
    ParadeStabilityInput(sequences=stability_utrs),
    ParadeStabilityConfig(device=DEVICE),
)
for r in stability.results:
    print(f"{r.sequence[:16]}...  log_ratio = {r.log_ratio:+.3f}")

## Part B — The differentiable objective (`parade-gradient`)

`parade-gradient` is the design primitive. Instead of a discrete sequence it takes **relaxed logits** of shape `(batch, length, 4)` (a soft, differentiable stand-in for a one-hot sequence) and returns the **gradient of a cell-type objective** with respect to those logits.

The objective is a sum of `ParadeGradientLossTerm`s. Each term names a `cell_type` and a `direction`:
- `direction="max"` → minimizes `1 - sigmoid(activity)`, i.e. *pushes that cell's activity up*;
- `direction="min"` → minimizes `sigmoid(activity)`, i.e. *pushes it down*.

So "maximize `c2`, minimize `c6`" is a **cell-type-specificity** objective. One call gives you the loss, the per-design raw activities, and the gradient you need to improve the logits.

In [ ]:
display_api_reference("parade-gradient", "input", "run_parade_gradient")
display_api_reference("parade-gradient", "config", "run_parade_gradient")

# One differentiable evaluation of a single random candidate: maximize c2, minimize c6.
rng = np.random.default_rng(0)
logits = rng.normal(0, 1, size=(1, 50, 4)).tolist()  # (batch=1, length=50, 4)

grad_out = run_parade_gradient(
    ParadeGradientInput(logits=logits, temperature=1.0),
    ParadeGradientConfig(
        construct_type="utr5",
        loss_terms=[
            ParadeGradientLossTerm(cell_type="c2", direction="max"),
            ParadeGradientLossTerm(cell_type="c6", direction="min"),
        ],
        device=DEVICE,
    ),
)

g = np.array(grad_out.gradient)                 # (1, 50, 4): dLoss/dlogits
raw = grad_out.sample_metrics[0]                # per-design raw activities for the objective cells
print("loss           :", round(grad_out.loss, 4))
print("gradient shape :", g.shape)
print("raw c2 activity:", round(raw['c2'], 3), "| raw c6 activity:", round(raw['c6'], 3))

## Part C — Design cell-type-specific UTRs (Fast SeqProp)

Now we turn the gradient into a design loop, [Fast SeqProp](https://doi.org/10.1186/s12859-021-04437-5)-style: start from random relaxed logits, and repeatedly (1) ask `parade-gradient` for the gradient of the specificity objective, then (2) take an **Adam** step on the logits. Descending the loss raises on-target activity and lowers off-target activity, so the specificity `activity(on) - activity(off)` climbs. At the end we `argmax` the logits into a concrete DNA sequence.

In [ ]:
def decode(logits):
    """Decode (N, L, 4) relaxed logits to DNA strings by per-position argmax (A,C,G,T order)."""
    return ["".join(VOCAB[i] for i in row.argmax(-1)) for row in logits]


def design_specific_utrs(
    construct_type, on_target, off_target,
    seq_len=50, n_designs=8, steps=50, lr=0.5, seed=0, device=DEVICE, log_every=10,
):
    """Fast SeqProp design of UTRs whose activity is specific to `on_target` over `off_target`.

    Optimizes a batch of `n_designs` relaxed logit tensors in parallel with Adam, driving the
    parade-gradient specificity objective. Returns (history, designs):
      - history: mean (on - off) predicted activity across the batch, per step (the climb);
      - designs: the final decoded DNA sequences.
    """
    # Start near a uniform distribution over nucleotides (small logits -> flat softmax), so the
    # optimizer isn't biased toward any starting sequence.
    rng = np.random.default_rng(seed)
    logits = 0.1 * rng.standard_normal((n_designs, seq_len, 4))

    # Adam optimizer state (first/second moment estimates) and hyperparameters.
    m = np.zeros_like(logits)
    v = np.zeros_like(logits)
    b1, b2, eps = 0.9, 0.999, 1e-8

    # The specificity objective is fixed across steps: push on_target up, off_target down.
    config = ParadeGradientConfig(
        construct_type=construct_type,
        loss_terms=[
            ParadeGradientLossTerm(cell_type=on_target, direction="max"),
            ParadeGradientLossTerm(cell_type=off_target, direction="min"),
        ],
        device=device,
    )

    history = []
    for t in range(1, steps + 1):
        # 1) One differentiable evaluation of the whole batch. temperature=1.0 is a plain softmax
        #    relaxation of the logits into nucleotide probabilities.
        out = run_parade_gradient(ParadeGradientInput(logits=logits.tolist(), temperature=1.0), config)

        # 2) Read the gradient of the loss w.r.t. the logits, same shape as `logits`.
        grad = np.array(out.gradient)  # (n_designs, L, 4)

        # 3) Track the mean specificity for this step. `sample_metrics[i]` carries the raw activity
        #    of each objective cell for design i, so specificity_i = activity(on) - activity(off).
        spec = float(np.mean([sm[on_target] - sm[off_target] for sm in out.sample_metrics]))
        history.append(spec)

        # 4) Adam update. We DESCEND the loss (subtract the step); because loss falls as on-target
        #    activity rises and off-target falls, the specificity we track goes up.
        m = b1 * m + (1 - b1) * grad
        v = b2 * v + (1 - b2) * grad ** 2
        m_hat = m / (1 - b1 ** t)
        v_hat = v / (1 - b2 ** t)
        logits -= lr * m_hat / (np.sqrt(v_hat) + eps)

        if log_every and (t == 1 or t % log_every == 0):
            print(f"  step {t:3d} | mean {on_target}-{off_target} specificity = {spec:+.3f}")

    # Collapse the optimized soft logits to concrete sequences.
    return history, decode(logits)

### Design 5'UTRs specific to `c2` over `c6`

In [ ]:
hist_utr5, designs_utr5 = design_specific_utrs("utr5", on_target="c2", off_target="c6")
print("\nexample designed 5'UTR:", designs_utr5[0])

### Design 3'UTRs specific to `c2` over `c13`

In [ ]:
hist_utr3, designs_utr3 = design_specific_utrs("utr3", on_target="c2", off_target="c13", seq_len=240)  # 3'UTR training length
print("\nexample designed 3'UTR:", designs_utr3[0])

### The climb

Mean predicted specificity across the 8 designs, per Fast SeqProp step — both panels climb from ~0.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
# One line per construct type; markers every step so the trajectory is easy to read.
ax.plot(range(1, len(hist_utr5) + 1), hist_utr5, color="#2a7de1", marker="o", ms=3, lw=2, label="5'UTR  (c2 vs c6)")
ax.plot(range(1, len(hist_utr3) + 1), hist_utr3, color="#e07b39", marker="o", ms=3, lw=2, label="3'UTR  (c2 vs c13)")
ax.axhline(0.0, color="0.6", lw=1, ls="--", zorder=0)  # zero specificity reference
ax.set_xlabel("Fast SeqProp step")
ax.set_ylabel("mean predicted specificity  (on \u2212 off activity)")
ax.set_title("PARADE-guided UTR design: cell-type specificity climbs")
ax.legend(frameon=False)
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

### Verify the designs in discrete space

The loop optimizes *relaxed* logits; what ultimately matters is the discrete `argmax` sequence. We re-score the decoded designs with `parade-activity` and compare their specificity to random sequences of the same length — the designs should be far more specific.

In [ ]:
def discrete_specificity(construct_type, seqs, on, off):
    """Mean (on - off) activity for a list of concrete sequences, via parade-activity."""
    out = run_parade_activity(
        ParadeActivityInput(sequences=seqs),
        ParadeActivityConfig(construct_type=construct_type, cell_types=[on, off], device=DEVICE),
    )
    return float(np.mean([r.scores[on] - r.scores[off] for r in out.results]))


# Random-sequence baselines of the same length, for reference.
rng = np.random.default_rng(1)
random_utr5 = ["".join(rng.choice(list(VOCAB), 50)) for _ in range(8)]
random_utr3 = ["".join(rng.choice(list(VOCAB), 240)) for _ in range(8)]  # match the 240-nt designs

rows = [
    ("5'UTR c2-c6", discrete_specificity("utr5", designs_utr5, "c2", "c6"),
                    discrete_specificity("utr5", random_utr5, "c2", "c6")),
    ("3'UTR c2-c13", discrete_specificity("utr3", designs_utr3, "c2", "c13"),
                     discrete_specificity("utr3", random_utr3, "c2", "c13")),
]
header = ("panel", "designed", "random")
print(f"{header[0]:<16}{header[1]:>10}{header[2]:>10}")
for panel, designed, random_ in rows:
    print(f"{panel:<16}{designed:>10.3f}{random_:>10.3f}")

## Export

Every tool output exports to its supported formats (`json`, `csv` for the scorers).

In [ ]:
from pathlib import Path
from tempfile import mkdtemp

out_dir = Path(mkdtemp())
# export(name, export_path=...): name is the file stem, export_path the directory.
activity.export("parade_activity", export_path=out_dir, file_format="csv")
stability.export("parade_stability", export_path=out_dir, file_format="json")
print(sorted(p.name for p in out_dir.iterdir()))

---

**Recap.** `parade-activity` and `parade-stability` score UTRs; `parade-gradient` turns the activity model into a differentiable objective; and a short Fast SeqProp loop over that gradient designs UTRs whose predicted activity is specific to one cell line over another — verified to hold after decoding to discrete DNA. Swap `on_target`/`off_target` for any codes in the panel (5'UTR: `c1, c2, c4, c6, c17`; 3'UTR adds `c13`), tune `lr`/`steps`/`n_designs`, add more `ParadeGradientLossTerm`s for a multi-cell objective, or select the winners for stability with `parade-stability`.